# SILVER LAYER
### Purpose:
Load Bronze Data

Data Cleaning

Data Transformation

Data Validation

Save Silver Data

# Import Libraries
Importing the required Python libraries for data processing, cleaning, transformation, and analysis.

In [ ]:
import pandas as pd
import numpy as np

# Connect Google Drive
Connecting Google Drive to access the project data and save outputs across all Medallion layers.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# Load Bronze Layer
Loading the validated raw data from the Bronze Layer into the Silver Layer for further processing and transformation.

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Bronze/bronze_salesss.csv"
)

print("Bronze Data Loaded Successfully")

Bronze Data Loaded Successfully


# Data Quality Check
Performing data quality checks to identify missing values, duplicate records, incorrect data types, and potential data issues before transformation.

In [ ]:
print(f"Rows Before Cleaning: {len(df)}")

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())


Rows Before Cleaning: 821250

Missing Values
Date                0
Branch              0
Drug_Name           0
Sales_Quantity      0
Unit_Price          0
Unit_Cost           0
Expired_Quantity    0
Damaged_Quantity    0
dtype: int64

Duplicate Rows
0


# Data Cleaning








##   Remove spaces from column names




In [ ]:
df.columns = df.columns.str.strip()

## Standardize Drug Names


In [ ]:
df["Drug_Name"] = df["Drug_Name"].str.strip().str.title()

## Remove Extra Spaces

In [ ]:
df = df.apply(
    lambda col: col.str.strip()
    if col.dtype == "object"
    else col
)

## Validate Numeric Columns

In [ ]:
df = df[df["Sales_Quantity"] >= 0]
df = df[df["Expired_Quantity"] >= 0]
df = df[df["Damaged_Quantity"] >= 0]

# Data Transformation
Transforming raw operational data into meaningful business metrics that support decision-making and performance analysis

## Revenue
Calculating total revenue generated from drug sales by multiplying sales quantity by unit selling price.

In [ ]:
df["Revenue"] = (
    df["Sales_Quantity"]
    * df["Unit_Price"]
)

## Total Cost
Calculating the total cost of sold products based on sales quantity and unit cost.

In [ ]:
df["Total_Cost"] = (
    df["Sales_Quantity"]
    * df["Unit_Cost"]
)

## Expired Loss
Calculating financial losses caused by expired drugs that could not be sold.

In [ ]:
df["Expired_Loss"] = (
    df["Expired_Quantity"]
    * df["Unit_Cost"]
)


## Damaged Loss
Calculating losses resulting from damaged products during storage or handling.

In [ ]:
df["Damaged_Loss"] = (
    df["Damaged_Quantity"]
    * df["Unit_Cost"]
)

## Net Profit
Calculating the actual profit after subtracting product cost, expired losses, and damaged losses from revenue.



In [ ]:
df["Net_Profit"] = (
    df["Revenue"]
    - df["Total_Cost"]
    - df["Expired_Loss"]
    - df["Damaged_Loss"]
)

## Waste Quantity
Calculating the total quantity of wasted products including expired and damaged inventory.

In [ ]:
df["Waste_Quantity"] = (
    df["Expired_Quantity"]
    + df["Damaged_Quantity"]
)


## Waste Rate
Measuring the percentage of wasted inventory relative to total sales quantity.

In [ ]:
df["Waste_Rate"] = np.where(
    df["Sales_Quantity"] == 0,
    0,
    (
        df["Waste_Quantity"]
        / df["Sales_Quantity"]
    )
)

## Profit Margin
Calculating the profitability ratio to evaluate how efficiently each product generates profit from sales.

In [ ]:
df["Profit_Margin"] = np.where(
    df["Revenue"] == 0,
    0,
    (
        df["Net_Profit"]
        / df["Revenue"]
    )
)

## Seasonality Analysis
Creating time-based attributes such as year, month, and month name to support seasonal trend analysis and demand forecasting.

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True
)
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()

# Validation
Validating the transformed dataset to ensure calculations are accurate and business rules are correctly applied.

## Check Missing Values

In [ ]:
df.isnull().sum()

,0
Date,0
Branch,0
Drug_Name,0
Sales_Quantity,0
Unit_Price,0
Unit_Cost,0
Expired_Quantity,0
Damaged_Quantity,0
Revenue,0
Total_Cost,0


In [ ]:
df.isnull().sum()

,0
Date,0
Branch,0
Drug_Name,0
Sales_Quantity,0
Unit_Price,0
Unit_Cost,0
Expired_Quantity,0
Damaged_Quantity,0
Revenue,0
Total_Cost,0


## Check Duplicate

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


## Check Unique Drug Names

In [ ]:
print("Unique Drugs:", df["Drug_Name"].nunique())
print(df["Drug_Name"].unique())

Unique Drugs: 50
['Paracetamol 500Mg' 'Amoxicillin 500Mg' 'Azithromycin 250Mg'
 'Metformin 850Mg' 'Atorvastatin 20Mg' 'Aspirin 81Mg' 'Ceftriaxone 1G'
 'Ibuprofen 400Mg' 'Omeprazole 20Mg' 'Salbutamol Inhaler'
 'Insulin Glargine' 'Insulin Regular' 'Vitamin D3 1000Iu'
 'Clopidogrel 75Mg' 'Losartan 50Mg' 'Enoxaparin 40Mg'
 'Hydrocortisone Cream' 'Diclofenac 50Mg' 'Levothyroxine 50Mcg'
 'Ciprofloxacin 500Mg' 'Pantoprazole 40Mg' 'Fluconazole 150Mg'
 'Warfarin 5Mg' 'Furosemide 40Mg' 'Lisinopril 10Mg' 'Prednisolone 5Mg'
 'Amiodarone 200Mg' 'Montelukast 10Mg' 'Sertraline 50Mg'
 'Alprazolam 0.5Mg' 'Insulin Aspart' 'Calcium Carbonate' 'Iron Supplement'
 'Cough Syrup' 'Antihistamine Syrup' 'Multivitamins' 'Antacid Suspension'
 'Glucose Test Strips' 'Syringes 5Ml' 'Iv Cannula' 'Nebulizer Mask'
 'Ors Sachets' 'Zinc Tablets' 'Antiseptic Solution'
 'Blood Pressure Monitor' 'Glucometer' 'Face Masks' 'Hand Sanitizer'
 'Thermometer Digital' 'Baby Formula']


## Check Unique Branch


In [ ]:
print("Unique Branches:", df["Branch"].nunique())
print(df["Branch"].unique())

Unique Branches: 15
['Branch_01' 'Branch_02' 'Branch_03' 'Branch_04' 'Branch_05' 'Branch_06'
 'Branch_07' 'Branch_08' 'Branch_09' 'Branch_10' 'Branch_11' 'Branch_12'
 'Branch_13' 'Branch_14' 'Branch_15']


## Check Revenue Logic
Validate that Revenue has been calculated correctly using sales quantity and unit price.

In [ ]:
revenue_check = (
    df["Revenue"]
    ==
    df["Sales_Quantity"] * df["Unit_Price"]
)

print("Revenue Calculation Correct:",
      revenue_check.all())

Revenue Calculation Correct: True


## Check Net Profit Logic
Validate that Net Profit has been calculated correctly according to the business formula.

In [ ]:
profit_check = (
    df["Net_Profit"]
    ==
    df["Revenue"]
    - df["Total_Cost"]
    - df["Expired_Loss"]
    - df["Damaged_Loss"]
)

print("Net Profit Calculation Correct:",
      profit_check.all())

Net Profit Calculation Correct: True


## Check Cost vs Price
Identify products where cost exceeds selling price, which may indicate pricing issues or potential losses.


In [ ]:
loss_products = (
    df["Unit_Cost"]
    >
    df["Unit_Price"]
).sum()

print("Records With Cost Greater Than Price:",
      loss_products)

Records With Cost Greater Than Price: 49275


## Check Negative Values

In [ ]:
numeric_cols = [
    "Sales_Quantity",
    "Unit_Price",
    "Unit_Cost",
    "Expired_Quantity",
    "Damaged_Quantity"
]

for col in numeric_cols:
    print(
        col,
        (df[col] < 0).sum()
    )

Sales_Quantity 0
Unit_Price 0
Unit_Cost 0
Expired_Quantity 0
Damaged_Quantity 0


## Validate Date Range

In [ ]:
print("Min Date:",
      df["Date"].min())

print("Max Date:",
      df["Date"].max())

Min Date: 2021-01-01 00:00:00
Max Date: 2023-12-31 00:00:00


# Silver Layer Summary
Summarizing the final cleaned and enriched dataset before storing it in the Silver Layer.

In [ ]:
df.describe()


,Date,Sales_Quantity,Unit_Price,Unit_Cost,Expired_Quantity,Damaged_Quantity,Revenue,Total_Cost,Expired_Loss,Damaged_Loss,Net_Profit,Waste_Quantity,Waste_Rate,Profit_Margin,Year,Month
count,821250,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000,821250.000000
mean,2022-07-02 00:00:00.000000256,39.410088,83.184263,58.651310,0.320597,0.267714,3279.096451,2049.782817,20.426549,12.398008,1196.489077,0.588311,0.017439,0.282083,2022.000000,6.526027
min,2021-01-01 00:00:00,1.000000,10.000000,5.040000,0.000000,0.000000,16.150000,10.520000,0.000000,0.000000,-3375.400000,0.000000,0.000000,-0.291932,2021.000000,1.000000
25%,2021-10-01 00:00:00,10.000000,46.580000,32.150000,0.000000,0.000000,684.300000,483.500000,0.000000,0.000000,156.000000,0.000000,0.000000,0.263921,2021.000000,4.000000
50%,2022-07-02 00:00:00,16.000000,82.770000,57.200000,0.000000,0.000000,1290.450000,910.900000,0.000000,0.000000,332.570000,0.000000,0.000000,0.296801,2022.000000,7.000000
75%,2023-04-02 00:00:00,22.000000,119.020000,82.490000,0.000000,0.000000,2266.380000,1619.857500,0.000000,0.000000,604.120000,0.000000,0.000000,0.331257,2023.000000,10.000000
max,2023-12-31 00:00:00,3652.000000,173.930000,198.030000,16.000000,128.000000,530054.560000,296028.480000,1612.000000,10063.780000,223972.960000,128.000000,0.250000,0.500000,2023.000000,12.000000
std,NaN,104.700321,42.442928,31.666353,0.889327,2.202122,9928.543364,5478.157177,66.349956,113.821118,4415.831268,2.351480,0.046229,0.121998,0.816497,3.447853


# Save Silver Layer
Saving the cleaned and transformed dataset into the Silver Layer for future analytical and reporting purposes.

In [ ]:
df.to_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Silver/silver_salesss.csv",
    index=False
)

